# 핸즈온 02 — CCTV/도로 이미지 멀티모달 분석

**소요 시간**: 80~90분
**학습 목표**:
1. Gemini 3 시리즈로 **차량/보행자 카운팅 + 좌표 추출**을 실행한다
2. **Pydantic 스키마**로 응답 구조를 강제하고, ITS 운영시스템과 연동 가능한 JSON을 생성한다
3. **media_resolution** 파라미터로 토큰·정확도 트레이드오프를 제어한다
4. 도로 파손(포트홀) 이미지 분류 시나리오로 응용한다

> **중요**: 강의 자료의 멀티모달 핸즈온은 `Gemini 1.5 Pro / 2.0 Flash`를 권장하지만, **두 모델 모두 셧다운(1.5)되었거나 셧다운 예정(2.0 Flash, 2026-06-01)**입니다. 본 핸즈온은 **Gemini 3 시리즈**로 작성되었습니다.

## 2-1. 환경 셋업

In [ ]:
!pip install -q -U google-genai pydantic pillow requests

In [ ]:
import os, json, time
from io import BytesIO
import requests
from PIL import Image, ImageDraw, ImageFont
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import Literal

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except ImportError:
    pass

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
print("✅ Client ready")

## 2-2. 테스트 이미지 — 도로/교차로 장면

공개 라이선스(Wikimedia Commons / Unsplash)에서 ITS 운영자가 보는 장면과 유사한 이미지를 가져옵니다.

In [ ]:
def load_image(url, max_size=1280):
    r = requests.get(url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    img = Image.open(BytesIO(r.content)).convert("RGB")
    img.thumbnail((max_size, max_size))
    return img

# 1. 교차로 (보행자 + 차량 혼재)
URL_INTERSECTION = "https://upload.wikimedia.org/wikipedia/commons/thumb/d/d0/Shibuya_Crossing_2018-08.jpg/1280px-Shibuya_Crossing_2018-08.jpg"

# 2. 고속도로 정체
URL_HIGHWAY = "https://upload.wikimedia.org/wikipedia/commons/thumb/9/97/I-80_Eastshore_Freeway.jpg/1280px-I-80_Eastshore_Freeway.jpg"

img_intersection = load_image(URL_INTERSECTION)
img_highway = load_image(URL_HIGHWAY)

print(f"교차로:  {img_intersection.size}")
print(f"고속도로: {img_highway.size}")
img_intersection

## 2-3. 첫 시도 — 단순 카운팅

먼저 가장 단순한 방식으로 객체를 세어봅니다.

In [ ]:
SIMPLE_PROMPT = """이 이미지에서 다음을 세어주세요:
- 차량 (vehicle)
- 보행자 (pedestrian)
- 자전거 (bicycle)
- 이륜차 (motorcycle)

JSON으로 출력하세요:
{"vehicles": N, "pedestrians": N, "bicycles": N, "motorcycles": N}"""

resp = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[SIMPLE_PROMPT, img_intersection],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        thinking_config=types.ThinkingConfig(thinking_level="low"),
    ),
)
print(json.dumps(json.loads(resp.text), indent=2, ensure_ascii=False))

## 2-4. Pydantic 스키마로 응답 구조 강제

JSON을 프롬프트로 "비는" 시대는 끝났습니다. Pydantic 모델을 직접 전달하면 SDK가 스키마를 강제합니다.

In [ ]:
class TrafficCount(BaseModel):
    """이미지의 교통 객체 카운트."""
    vehicles: int = Field(description="모든 차량 (승용차, 트럭, 버스 합산)")
    pedestrians: int = Field(description="보행 중인 사람")
    bicycles: int = Field(description="자전거")
    motorcycles: int = Field(description="오토바이/이륜차")
    overall_density: Literal["light", "moderate", "heavy"] = Field(
        description="전체적인 교통 밀도"
    )
    notes: str = Field(description="ITS 운영자가 알아야 할 특이사항 1줄", max_length=120)


resp = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=["이 이미지를 분석해주세요.", img_intersection],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=TrafficCount,
        thinking_config=types.ThinkingConfig(thinking_level="medium"),
    ),
)

# Pydantic 인스턴스로 직접 받기
result: TrafficCount = resp.parsed
print(f"차량:    {result.vehicles}")
print(f"보행자:  {result.pedestrians}")
print(f"자전거:  {result.bicycles}")
print(f"이륜차:  {result.motorcycles}")
print(f"밀도:    {result.overall_density}")
print(f"메모:    {result.notes}")

> **포인트**: `resp.parsed`로 받으면 Pydantic 인스턴스가 그대로 들어옵니다. 운영 시스템과 연동할 때 타입 안정성이 보장됩니다.

## 2-5. Bounding Box — 객체 위치 추출

Gemini는 객체의 위치를 `[ymin, xmin, ymax, xmax]` 형식의 **0~1000 정규화 좌표**로 반환합니다. 이걸 픽셀 좌표로 변환해 시각화합니다.

In [ ]:
class DetectedObject(BaseModel):
    label: str = Field(description="객체 종류: vehicle, pedestrian, bicycle, motorcycle 중 하나")
    box_2d: list[int] = Field(
        description="[ymin, xmin, ymax, xmax] 형식의 0-1000 정규화 좌표",
        min_length=4, max_length=4,
    )
    description: str = Field(description="간단한 설명 (예: 'white sedan', 'walking person')")


class DetectionResult(BaseModel):
    objects: list[DetectedObject]
    summary: str = Field(description="장면 요약 1줄")


DETECTION_PROMPT = """이 이미지의 모든 차량, 보행자, 자전거, 이륜차를 탐지하세요.
각 객체의 위치를 [ymin, xmin, ymax, xmax] 형식의 0~1000 정규화 좌표로 출력하세요.
가까이 있는 큰 객체 위주로 최대 30개까지 탐지하세요."""

resp = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[DETECTION_PROMPT, img_intersection],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=DetectionResult,
        thinking_config=types.ThinkingConfig(thinking_level="medium"),
    ),
)

detection: DetectionResult = resp.parsed
print(f"탐지된 객체: {len(detection.objects)}개")
print(f"요약: {detection.summary}\n")
for obj in detection.objects[:5]:
    print(f"  - {obj.label}: {obj.description}  box={obj.box_2d}")

### Bounding Box 시각화

In [ ]:
def draw_boxes(img, objects, color_map=None):
    """0-1000 정규화 좌표 → 픽셀 좌표 변환 후 박스 그리기."""
    if color_map is None:
        color_map = {
            "vehicle": "#e74c3c",
            "pedestrian": "#3498db",
            "bicycle": "#2ecc71",
            "motorcycle": "#f39c12",
        }
    img_copy = img.copy()
    draw = ImageDraw.Draw(img_copy)
    W, H = img.size

    for obj in objects:
        ymin, xmin, ymax, xmax = obj.box_2d
        # 0-1000 → 픽셀
        x1 = int(xmin / 1000 * W)
        y1 = int(ymin / 1000 * H)
        x2 = int(xmax / 1000 * W)
        y2 = int(ymax / 1000 * H)
        color = color_map.get(obj.label, "#ffffff")
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        draw.text((x1+3, y1+3), obj.label, fill=color)
    return img_copy


vis = draw_boxes(img_intersection, detection.objects)
vis

> **체크 포인트**
>
> 1. 박스가 객체에 잘 맞는가? 크게 어긋난 객체가 있다면 어떤 종류?
> 2. 작은 객체(멀리 있는 보행자 등)는 잘 잡혔는가?
> 3. 객체 라벨링이 일관적인가? (예: 같은 종류를 vehicle/car 혼용하지 않는가)
>
> Gemini의 좌표 정확도는 객체가 작거나 겹쳐있을수록 떨어집니다. **Pro 모델이 Flash 대비 좌표 정확도가 더 좋다**고 알려져 있지만, 비용이 매우 다릅니다.

## 2-6. media_resolution — 토큰 vs 정확도 트레이드오프

Gemini 3는 이미지당 토큰 수를 직접 제어할 수 있습니다.

| 설정 | 토큰/이미지 | 용도 |
|---|---|---|
| `media_resolution_low` | 280 | 단순 분류, 낮은 비용 |
| `media_resolution_medium` | 560 | 일반 (기본값) |
| `media_resolution_high` | 1120 | OCR, 작은 글자, 정밀 분석 |

In [ ]:
def detect_with_resolution(img, resolution):
    t0 = time.time()
    config = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=DetectionResult,
        thinking_config=types.ThinkingConfig(thinking_level="low"),
        media_resolution=resolution,
    )
    resp = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=[DETECTION_PROMPT, img],
        config=config,
    )
    return resp, time.time() - t0


for res in ["media_resolution_low", "media_resolution_medium", "media_resolution_high"]:
    try:
        resp, lat = detect_with_resolution(img_intersection, res)
        u = resp.usage_metadata
        result: DetectionResult = resp.parsed
        print(f"\n=== {res} ===")
        print(f"  Latency:      {lat:.2f}s")
        print(f"  Input tokens: {u.prompt_token_count:,}")
        print(f"  탐지 객체:    {len(result.objects)}개")
    except Exception as e:
        print(f"\n❌ {res}: {e}")
    time.sleep(1.5)

> **관찰**: 해상도가 올라가면 입력 토큰이 증가하지만 더 많은 객체를 잡거나 작은 객체의 좌표 정확도가 올라갑니다. **번호판 같이 작은 글자를 읽어야 한다면 `high`가 필수**입니다.

## 2-7. 응용 시나리오 — 도로 파손(포트홀) 자동 분류

ITS 도로관리 영역의 실제 워크로드입니다. 도로 파손 이미지를 분류하고 보수 우선순위를 제안하게 합니다.

In [ ]:
# Wikimedia Commons - 포트홀 이미지 (공개 라이선스)
URL_POTHOLE = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/35/Pothole.jpg/1024px-Pothole.jpg"

img_pothole = load_image(URL_POTHOLE)
img_pothole

In [ ]:
class RoadDamageReport(BaseModel):
    damage_type: Literal["pothole", "crack_longitudinal", "crack_transverse",
                          "crack_alligator", "rutting", "patching", "other"] = Field(
        description="도로 파손 유형"
    )
    severity: Literal["low", "medium", "high", "critical"] = Field(
        description="심각도. critical은 즉시 통제가 필요한 수준"
    )
    estimated_size_cm: int = Field(
        description="파손부 추정 크기(cm). 정확한 측정이 어려우면 보이는 단서로 추정"
    )
    safety_risk: str = Field(
        description="이 파손이 야기할 수 있는 안전 위험 1줄 설명",
        max_length=150,
    )
    repair_priority_days: int = Field(
        description="권장 보수 시한(일). 0=즉시, 7=1주 내, 30=한 달 내",
        ge=0, le=365,
    )
    repair_method: str = Field(
        description="권장 보수 방법 (예: '냉간 아스팔트 패칭', '절삭 후 재포장')",
        max_length=80,
    )


PROMPT = """당신은 도로 유지관리 점검관입니다. 이 이미지의 도로 파손 상태를 평가하고
보수 계획을 제안해주세요."""

resp = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[PROMPT, img_pothole],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=RoadDamageReport,
        thinking_config=types.ThinkingConfig(thinking_level="medium"),
    ),
)

report: RoadDamageReport = resp.parsed
print(f"파손 유형:     {report.damage_type}")
print(f"심각도:        {report.severity}")
print(f"추정 크기:     {report.estimated_size_cm} cm")
print(f"안전 위험:     {report.safety_risk}")
print(f"보수 시한:     {report.repair_priority_days}일 내")
print(f"보수 방법:     {report.repair_method}")

## 2-8. 실전 패턴 — 배치 처리 함수

여러 이미지를 일괄 처리하는 패턴입니다. 운영 시스템에 그대로 가져갈 수 있습니다.

In [ ]:
def analyze_batch(image_urls, model="gemini-3.1-flash-lite-preview"):
    """여러 이미지를 일괄 분석. ITS 운영시스템 통합용 패턴."""
    results = []
    for url in image_urls:
        try:
            img = load_image(url, max_size=1024)
            resp = client.models.generate_content(
                model=model,
                contents=["이 이미지의 교통 객체를 분석하세요.", img],
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=TrafficCount,
                    thinking_config=types.ThinkingConfig(thinking_level="low"),
                ),
            )
            r: TrafficCount = resp.parsed
            results.append({"url": url, "ok": True, "data": r.model_dump()})
        except Exception as e:
            results.append({"url": url, "ok": False, "error": str(e)[:80]})
        time.sleep(1.0)
    return results


URLS = [URL_INTERSECTION, URL_HIGHWAY]
batch_results = analyze_batch(URLS)
for r in batch_results:
    if r["ok"]:
        d = r["data"]
        print(f"\n{r['url'][-50:]}")
        print(f"  vehicles={d['vehicles']}, pedestrians={d['pedestrians']}, density={d['overall_density']}")
    else:
        print(f"\n❌ {r['error']}")

## 2-9. 도전 과제

다음 중 하나를 선택해 구현해보세요. 시간이 남는 조는 둘 다 시도하세요.

### 과제 A — 시간대 추정 + 야간 검출
다음 정보를 추가로 추출하는 Pydantic 스키마를 설계하세요:
- 추정 시간대 (주간/야간/석양/새벽)
- 가시성 (clear / rain / fog / snow)
- 도로 노면 상태 (dry / wet / snowy)

### 과제 B — 차종별 세분화 카운팅
승용차 / SUV / 화물차 / 버스 / 이륜차로 세분화해 카운팅하고, 각 차종별 비율도 출력하는 스키마를 설계하세요. **혼잡 통행료 책정 시뮬레이션**에 활용 가능한 형식이어야 합니다.

> 힌트: Pydantic의 `Literal`, `dict[str, int]`, `Field(description=...)`을 적극 활용하세요.

## 2-10. 정리

- ✅ Pydantic 스키마로 응답 구조 강제 (`response_schema`)
- ✅ Bounding Box 좌표 추출 + 시각화
- ✅ `media_resolution` 으로 토큰·정확도 제어
- ✅ 도로 파손 분류 응용 시나리오
- ✅ 배치 처리 패턴

**핵심 교훈**: 운영 시스템과 연동할 때는 **반드시 스키마를 강제**해야 합니다. 프롬프트에 "JSON으로 줘"라고 비는 방식은 production 환경에서 깨집니다.

다음 핸즈온(`03_long_context_caching.ipynb`)에서는 1M 컨텍스트 윈도우를 활용한 대용량 데이터 분석을 다룹니다.